# Punto 3 — RAG con Ollama: asistente de consulta sobre derechos y normativa colombiana

**Base de contexto (3 documentos):**
1. Constitución Política de Colombia (edición Corte Constitucional 2024) — el mismo corpus de los Puntos 1 y 2.
2. **Ley 1755 de 2015** — regula el derecho fundamental de petición. *Nota:* sustituye los artículos 13–33 del CPACA (Ley 1437 de 2011); por eso un "Art. 14" citado de este documento es el art. 14 del CPACA tal como lo dejó esta ley.
3. **Ley 1581 de 2012** — protección de datos personales (hábeas data).

Las dos leyes desarrollan los artículos **23** (petición) y **15** (datos) de la Constitución, así que hay preguntas **cruzadas** entre documentos.

**Pipeline:** PDFs → limpieza → *chunking* (por artículo) → embeddings (Ollama) → Chroma + BM25 → recuperación **híbrida y balanceada por documento** → umbral de abstención → prompt con contexto → generación (Ollama) → fuentes añadidas por código → Gradio.

**Evaluación:** recuperación (Hit@k, Recall@k, MRR, cobertura en preguntas cruzadas), generación (dato clave, similitud, fidelidad con juez LLM, rúbrica manual), **RAG vs sin RAG**, preguntas **sin respuesta** (lejanas y cercanas) y ablaciones (chunking, tipo de recuperación, k).

In [1]:
import sys, os, subprocess, shutil, time
EN_COLAB = "google.colab" in sys.modules
os.environ["ANONYMIZED_TELEMETRY"] = "False"        # debe ir ANTES de importar chromadb (quita los avisos de telemetría)
if EN_COLAB:                                        # Colab: instalar dependencias y Ollama
    print("Instalando dependencias y Ollama en Colab (2-3 min)…")
    subprocess.run("apt-get install -y -q zstd && pip install -q ollama chromadb pypdf gradio "
                   "&& curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

In [ ]:
import re, json, pickle, hashlib, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
import ollama
import chromadb
from chromadb.config import Settings
from pypdf import PdfReader

def buscar_carpeta_entregable():
    """Carpeta que contiene el PDF de la Constitución. Orden: variable de entorno ENTREGABLE_DIR -> carpeta
    del notebook -> sitios típicos del home -> (último recurso) una partición de Windows montada."""
    import glob
    cands = []
    if os.environ.get("ENTREGABLE_DIR"):
        cands.append(Path(os.environ["ENTREGABLE_DIR"]).expanduser())
    h = Path.home()
    cands += [Path.cwd(), Path.cwd().parent, h / "Entregable Transformers"]
    cands += [h / d / "IA Generativa" / "Entregable Transformers" for d in ("Escritorio", "Desktop", "Documentos", "Documents")]
    cands += [Path(p) for p in glob.glob("/media/*/*/Users/*/OneDrive/Escritorio/IA Generativa/Entregable Transformers")
                                + glob.glob("/mnt/*/Users/*/OneDrive/Escritorio/IA Generativa/Entregable Transformers")]
    for p in cands:
        if p.is_dir() and list(p.glob("Constituci*2024*.pdf")):
            return p
    raise FileNotFoundError("No encontré la carpeta con Constitución_2024.pdf. Abre el notebook desde esa carpeta "
                            "o define la variable de entorno ENTREGABLE_DIR con su ruta.")

if EN_COLAB:
    RUTA = Path("/content"); DOCS_DIR = RUTA / "documentos_rag"; DOCS_DIR.mkdir(exist_ok=True)
    if not list(RUTA.glob("Constituci*2024*.pdf")):
        from google.colab import files
        print("Sube los 3 PDF (Constitución y las dos leyes):")
        for nombre in files.upload():
            destino = RUTA if nombre.lower().startswith("constituci") else DOCS_DIR
            Path(nombre).replace(destino / nombre)
else:
    RUTA = buscar_carpeta_entregable()
    DOCS_DIR = RUTA / "documentos_rag"; DOCS_DIR.mkdir(exist_ok=True)
SALIDAS = RUTA / "salidas"; SALIDAS.mkdir(exist_ok=True)
print("Carpeta de trabajo:", RUTA)
print("Documentos extra en:", DOCS_DIR, "->", [p.name for p in DOCS_DIR.iterdir()])

Carpeta de trabajo: /home/isabellap/Escritorio/Entregable Transformers
Documentos extra en: /home/isabellap/Escritorio/Entregable Transformers/documentos_rag -> ['Ley_1755_de_2015.pdf', 'Ley_1581_2012.pdf', 'Constitución_2024.pdf']


## 1. Servidor Ollama y modelos
Los modelos se cambian aquí:
- `LLM_MODEL`: generador (alternativas: `qwen2.5:3b`, `phi3`, `llama3.2:1b`).
- `EMB_MODEL`: embeddings multilingües (mejor en español). Alternativa ligera: `nomic-embed-text`.

In [3]:
LLM_MODEL = "llama3.2:3b"
EMB_MODEL = "bge-m3"

def encontrar_ollama():
    """Ruta del binario de Ollama: PATH o ubicaciones típicas en Linux."""
    exe = shutil.which("ollama")
    if exe:
        return exe
    return next((p for p in ("/usr/local/bin/ollama", "/usr/bin/ollama", "/opt/ollama/bin/ollama") if Path(p).is_file()), None)

OLLAMA_EXE = encontrar_ollama()
print("Ejecutable de Ollama:", OLLAMA_EXE)

def asegurar_ollama(espera=25):
    """Comprueba que el servidor responda; si no, intenta lanzarlo (`ollama serve`)."""
    try:
        ollama.list(); print("Ollama ya está corriendo ✔"); return
    except Exception:
        pass
    if OLLAMA_EXE is None:
        raise RuntimeError("No encuentro ollama. Instálalo con: curl -fsSL https://ollama.com/install.sh | sh  (y reinicia el kernel).")
    subprocess.Popen([OLLAMA_EXE, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(espera):
        time.sleep(1)
        try:
            ollama.list(); print("Servidor Ollama iniciado ✔"); return
        except Exception:
            pass
    raise RuntimeError("Ollama no respondió. Ábrelo manualmente y vuelve a ejecutar esta celda.")

def asegurar_modelo(nombre):
    instalados = {m.model for m in ollama.list().models}
    if nombre in instalados or f"{nombre}:latest" in instalados:
        print(f"✔ {nombre} ya está descargado")
    else:
        print(f"Descargando {nombre} (puede tardar varios minutos)…"); ollama.pull(nombre)

asegurar_ollama()
for m in (LLM_MODEL, EMB_MODEL):
    asegurar_modelo(m)

Ejecutable de Ollama: /usr/local/bin/ollama
Ollama ya está corriendo ✔
✔ llama3.2:3b ya está descargado
✔ bge-m3 ya está descargado


In [ ]:
# validacion del uso de GPU 
ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": "hola"}], options={"num_predict": 1})
if OLLAMA_EXE:
    print(subprocess.run([OLLAMA_EXE, "ps"], capture_output=True, text=True).stdout)

NAME           ID              SIZE      PROCESSOR         CONTEXT    RUNNER      UNTIL              
llama3.2:3b    a80c4f17acd5    2.4 GB    6%/94% CPU/GPU    4096       llamacpp    4 minutes from now    



## 2. Carga y limpieza de documentos
- La **Constitución** usa la limpieza específica (omite créditos e índice, quita encabezados de página).
- Las **demás normas** usan una limpieza genérica: se quitan URLs, fechas de impresión, marcadores de página (`13/204`) y líneas repetidas en muchas páginas (encabezados/pies). Después se **recorta todo lo que hay tras la firma** ("Dada en Bogotá…"): algunos PDF (p. ej. SUIN‑Juriscol) traen la ley seguida de la sentencia de constitucionalidad y notas, que contaminarían el índice con artículos que no son de la ley.
- La Constitución **nunca** se carga dos veces

In [ ]:
USAR_PDFS_DE_CLASE = False

PAG_INICIO = 20
NOTA = re.compile(
    r"(?:Inciso|Incisos|Parágrafo|Numeral|Literal|Aparte|Apartes|Expresión|Artículo|Título|Capítulo|Texto|Frase|Palabra|Palabras)s?"
    r"\s(?:[^.]|No\.){0,90}?\b(?:adicionad|modificad|derogad|subrogad|sustituid|declarad|suprimid|incluid|reformad|adicionó)\w*"
    r"(?:[^.]|No\.){0,200}?\d{4}\.\s*", re.I)

NOMBRE_LIBRO = "Constitución Política de Colombia"
RE_GUARDIANA = re.compile(r"^\s*(?:\d{1,3}\s*)?Guardiana de la integridad y supremacía de la Constitución\s*\d{0,3}\s*")
RE_PREFIJO = re.compile(r"Guardiana|Preámbulo|(?:TÍTULO|Título)\s+(?:[IVXL]+|TRANSITORIO):|^\s*\d{1,3}\b|\d{1,3}\s*$")

def limpiar_pagina_const(txt):
    """Quita el encabezado de cada página. Es robusto a la versión de pypdf (que puede pegar el
    encabezado con la primera línea del texto): corta hasta el nombre del libro si el prefijo parece encabezado."""
    t = RE_GUARDIANA.sub("", txt)
    k = t.find(NOMBRE_LIBRO)
    if 0 <= k < 350:
        pref = t[:k].strip()
        if pref == "" or (RE_PREFIJO.search(pref) and not re.search(r"\.\s", pref)):
            t = t[k + len(NOMBRE_LIBRO):]
    return t.lstrip("\n ")

def unir_guiones(t):
    """Algunas versiones de pypdf dejan 'participa-ción' sin salto de línea. Se une si la palabra completa
    existe en el texto o si alguna de las dos mitades nunca aparece como palabra suelta;
    así se conservan guiones legítimos como 'político-administrativo'."""
    cnt = Counter(re.findall(r"\w+", re.sub(r"\w+-\w+", " ", t.lower())))   # palabras sueltas, sin las compuestas con guion
    def f(m):
        a, b = m.group(1), m.group(2)
        if cnt[(a + b).lower()] >= 1 or cnt[a.lower()] == 0 or cnt[b.lower()] == 0:
            return a + b
        return m.group(0)
    return re.sub(r"\b([a-záéíóúñü]{2,})-([a-záéíóúñü]{2,})\b", f, t, flags=re.I)

def es_encabezado(s):
    letras = [c for c in s if c.isalpha()]
    return len(letras) >= 3 and all(c.isupper() for c in letras) and len(s) < 120

def limpiar_texto(raw):
    """Une líneas de un mismo párrafo, reconstruye palabras partidas y quita notas editoriales."""
    t = raw.replace("\u00ad", "")
    t = re.sub(r"Guardiana de la integridad y supremacía de la Constitución\s*\d{0,3}", " ", t)   # encabezado suelto
    t = re.sub(r"(?<=\w) ?-[ \t]*\n[ \t]*(?=[a-záéíóúñü])", "", t)
    t = NOTA.sub("", t)
    t = re.sub(r"(?:Modificado|Adicionado|Derogado|Subrogado|Sustituido)s? por el art[ií]culo(?:[^.]|No\.){0,140}?\d{4}\.\s*", "", t)
    t = re.sub(r"(?m)^\s*\d{1,3}\s*$", "", t)
    t = re.sub(r"<[^<>]{0,250}>", "", t)
    t = unir_guiones(re.sub(r"[ \t]+", " ", t))
    parrafos, act, prev_head = [], "", False
    for s in (l.strip() for l in t.split("\n")):
        if not s:
            continue
        cabeza = es_encabezado(s) or re.match(r"(ARTÍCULO|ART[ÍI]CULO|PARÁGRAFO|CAPÍTULO|TÍTULO)\b", s, re.I)
        corte = cabeza or prev_head or (act and act[-1] in ".:;!?" and s[0].isupper())
        if corte and act:
            parrafos.append(act); act = s
        else:
            act = (act + " " + s) if act else s
        prev_head = es_encabezado(s)
    if act:
        parrafos.append(act)
    return "\n".join(parrafos)

def leer_constitucion(path):
    pags = PdfReader(str(path)).pages[PAG_INICIO:]
    return limpiar_texto("\n".join(limpiar_pagina_const(p.extract_text() or "") for p in pags))

# ---------- Limpieza genérica para normas ----------
def recortar_tras_firma(t, extra=450):
    """Corta todo lo posterior a la firma de la ley ('Dada en Bogotá…'): sentencias, notas y concordancias."""
    m = re.search(r"Dada en Bogot[aá]", t)
    return t[:m.start() + extra] if m else t

def quitar_ruido_pagina(pg):
    pg = re.sub(r"https?://\S+", " ", pg)                                                # URLs
    pg = re.sub(r"\d{1,2}/\d{1,2}/\d{2,4},? \d{1,2}:\d{2}[^\n]{0,80}", " ", pg)         # fecha-hora de impresión (SUIN)
    pg = re.sub(r"(?m)^\s*\d{1,3}/\d{1,3}\s*$", " ", pg)                                 # marcadores de página "13/204"
    return pg

def leer_generico(path):
    if path.suffix.lower() == ".txt":
        return recortar_tras_firma(limpiar_texto(quitar_ruido_pagina(path.read_text(encoding="utf-8", errors="ignore"))))
    paginas = [quitar_ruido_pagina(p.extract_text() or "") for p in PdfReader(str(path)).pages]
    # líneas cortas que aparecen en ≥ 30% de las páginas = encabezados/pies repetidos
    cnt = {}
    for pg in paginas:
        L = [l.strip() for l in pg.split("\n") if l.strip()]
        for l in set(L[:3] + L[-3:]):
            if len(l) < 110:
                cnt[l] = cnt.get(l, 0) + 1
    repetidas = {l for l, c in cnt.items() if c >= max(3, 0.3 * len(paginas))}
    limpias = ["\n".join(l for l in pg.split("\n") if l.strip() not in repetidas) for pg in paginas]
    return recortar_tras_firma(limpiar_texto("\n".join(limpias)))

# ---------- Cargar todos los documentos ----------
def cargar_documentos():
    docs = {}
    const = list(RUTA.glob("Constituci*2024*.pdf"))
    if const:
        docs[const[0].name] = leer_constitucion(const[0])
    extras = sorted(list(DOCS_DIR.glob("*.pdf")) + list(DOCS_DIR.glob("*.txt")))
    extras = [p for p in extras if not re.match(r"(?i)constituci", p.name)]       # la Constitución ya se cargó con su limpieza propia
    if USAR_PDFS_DE_CLASE and len(extras) < 2:
        extras += sorted(RUTA.glob("Avances*.pdf")) + sorted(RUTA.glob("Transformer_Impacto*.pdf"))
    for p in extras:
        docs[p.name] = leer_generico(p)
    return docs

docs = cargar_documentos()
if len(docs) < 3:
    raise RuntimeError(f"Solo encontré {len(docs)} documento(s): {list(docs)}. El enunciado pide al menos 3: "
                       f"copia las dos leyes (PDF) en {DOCS_DIR}")
for k, v in docs.items():
    aviso = "  ⚠️ ¿incluye sentencias/notas? Debería tener < 100 mil caracteres" if (not k.lower().startswith("constituci") and len(v) > 200_000) else ""
    print(f"{k}: {len(v):,} caracteres, {len(re.findall(r'(?i)art[ií]culo', v))} menciones de 'artículo'{aviso}")

invalid pdf header: b'\t\n%PD'
EOF marker seems truncated
incorrect startxref pointer(1)
parsing for Object Streams


Constitución_2024.pdf: 454,507 caracteres, 702 menciones de 'artículo'
Ley_1581_2012.pdf: 33,341 caracteres, 40 menciones de 'artículo'
Ley_1755_de_2015.pdf: 22,695 caracteres, 29 menciones de 'artículo'


## 3. Chunking: por artículo (con subdivisión de los largos) o por tamaño fijo
Los textos legales ya están estructurados en artículos, así que el modo **`articulo`** crea un chunk por artículo (los que superan `max_size` se subdividen con solape y conservan la etiqueta y el encabezado `ARTÍCULO N (continuación)`). La **cita es exacta** porque cada chunk pertenece a un solo artículo. El modo **`fijo`** parte por tamaño y se usa como comparación en la ablación (sección 9).

In [6]:
RE_INICIO_ART = re.compile(r"(?m)^(?:ART[ÍI]CULO|Art[ií]culo)\s+(\d+)")      # solo encabezados de artículo (inicio de línea)

def chunkear(texto, size=800, overlap=120):
    """Parte por tamaño con solape, cortando en un límite natural. Devuelve [(posición_inicial, texto)]."""
    chunks, i = [], 0
    while i < len(texto):
        fin = min(i + size, len(texto))
        if fin < len(texto):                               # buscar un corte natural en el último 25% del chunk
            ventana = texto[i + int(size * 0.75): fin]
            corte = max(ventana.rfind("\n"), ventana.rfind(". "))
            if corte != -1:
                fin = i + int(size * 0.75) + corte + 1
        c = texto[i:fin].strip()
        if len(c) > 50:
            chunks.append((i, c))
        if fin >= len(texto):
            break
        i = max(fin - overlap, i + 1)
    return chunks

def etiqueta_fijo(texto_doc, inicio, chunk):
    """Etiqueta de artículo para chunks de tamaño fijo: el/los artículos que EMPIEZAN dentro del chunk,
    o el último artículo anterior si el chunk es continuación."""
    en_chunk = RE_INICIO_ART.findall(chunk)
    if en_chunk:
        return en_chunk[0] if en_chunk[0] == en_chunk[-1] else f"{en_chunk[0]}-{en_chunk[-1]}"
    previos = RE_INICIO_ART.findall(texto_doc[:inicio])
    return previos[-1] if previos else "—"

def chunks_por_articulo(texto, max_size=1200, overlap=150):
    """[(posición, texto, etiqueta)] con un chunk por artículo. Los encabezados de TÍTULO/CAPÍTULO que quedan
    al final de un artículo se pasan al artículo siguiente (son su contexto)."""
    marcas = [(m.start(), m.group(1)) for m in RE_INICIO_ART.finditer(texto)]
    if not marcas:
        return [(o, c, "—") for o, c in chunkear(texto, max_size, overlap)]
    unidades = [(0, texto[:marcas[0][0]], "—")] if marcas[0][0] > 0 else []
    for k, (ini, num) in enumerate(marcas):
        fin = marcas[k + 1][0] if k + 1 < len(marcas) else len(texto)
        unidades.append((ini, texto[ini:fin], num))
    out, pendiente = [], ""
    for ini, u, num in unidades:
        lineas = [l for l in u.strip().split("\n")]
        cola = []
        while len(lineas) > 1 and (es_encabezado(lineas[-1]) or re.match(r"(TÍTULO|CAPÍTULO)\b", lineas[-1])):
            cola.insert(0, lineas.pop())
        u = (pendiente + "\n" + "\n".join(lineas)).strip() if pendiente else "\n".join(lineas).strip()
        pendiente = "\n".join(cola)
        if len(u) <= max_size:
            if len(u) > 50:
                out.append((ini, u, num))
        else:
            for j, (o, c) in enumerate(chunkear(u, max_size, overlap)):
                if j > 0 and num != "—":
                    c = f"ARTÍCULO {num} (continuación). " + c
                out.append((ini + o, c, num))
    return out

def construir_chunks(docs, modo="articulo", size=1200, overlap=150):
    out = []
    for nombre, t in docs.items():
        if modo == "articulo":
            partes = chunks_por_articulo(t, size, overlap)
        else:
            partes = [(ini, c, etiqueta_fijo(t, ini, c)) for ini, c in chunkear(t, size, overlap)]
        for j, (_, c, art) in enumerate(partes):
            out.append({"id": f"{nombre}::{j}", "doc": nombre, "art": art, "texto": c})
    return out

CFG_CHUNK = dict(modo="articulo", size=1200, overlap=150)          # configuración principal
chunks = construir_chunks(docs, **CFG_CHUNK)
print("Total de chunks:", len(chunks), "| por documento:", pd.Series([c["doc"] for c in chunks]).value_counts().to_dict())
print("Longitud de chunks (caracteres): mediana", int(np.median([len(c['texto']) for c in chunks])), "| máx", max(len(c['texto']) for c in chunks))
ej = next(c for c in chunks if c["art"] == "86")
print(f"\n[{ej['doc']} · Art. {ej['art']}]\n{ej['texto'][:500]}")

Total de chunks: 735 | por documento: {'Constitución_2024.pdf': 657, 'Ley_1581_2012.pdf': 46, 'Ley_1755_de_2015.pdf': 32}
Longitud de chunks (caracteres): mediana 896 | máx 1229

[Constitución_2024.pdf · Art. 86]
ARTÍCULO 86. Toda persona tendrá acción de tutela para reclamar ante los jueces, en todo momento y lugar, mediante un procedimiento preferente y sumario, por sí misma o por quien actúe a su nombre, la protección inmediata de sus derechos constitucionales fundamentales, cuando quiera que estos resulten vulnerados o amenazados por la acción o la omisión de cualquier autoridad pública.
La protección consistirá en una orden para que aquel respecto de quien se solicita la tutela, actúe o se abstenga 


## 4. Embeddings + base vectorial (Chroma) + índice léxico (BM25)
- Los embeddings se guardan en un **caché en disco** (`salidas/cache_embeddings.pkl`).
- Cada índice (`Indice`) guarda: la colección de **Chroma** (búsqueda vectorial), la matriz de embeddings normalizados (para calcular distancias de cualquier chunk) y un **BM25** propio. El BM25 recupera coincidencias exactas (“pena de muerte”, “bancos de datos”) que los vectores a veces pierden.

In [7]:
client = chromadb.Client(Settings(anonymized_telemetry=False))      # base vectorial en memoria
CACHE_PATH = SALIDAS / "cache_embeddings.pkl"
_cache = pickle.load(open(CACHE_PATH, "rb")) if CACHE_PATH.exists() else {}

def _clave(t): return hashlib.md5((EMB_MODEL + "|" + t).encode("utf-8")).hexdigest()

def embed(textos, bs=16):
    """Embeddings con Ollama + caché. Devuelve una lista de vectores en el mismo orden que `textos`."""
    faltan = list(dict.fromkeys(t for t in textos if _clave(t) not in _cache))
    for i in range(0, len(faltan), bs):
        lote = faltan[i:i + bs]
        for t, e in zip(lote, ollama.embed(model=EMB_MODEL, input=lote)["embeddings"]):
            _cache[_clave(t)] = e
    if faltan:
        pickle.dump(_cache, open(CACHE_PATH, "wb"))
    return [_cache[_clave(t)] for t in textos]

# ---------- BM25 (búsqueda léxica) ----------
STOP = set("que los las del por con una para como son sus les este esta estos estas entre sobre segun cual cuales cuando donde "
           "quien hay sin ser tiene tienen dice cuanto cuantos cual debe deben puede pueden otro otra todo toda todos todas".split())
def tok(t):
    t = unicodedata.normalize("NFD", t.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")                 # sin tildes
    ws = [w for w in re.findall(r"[a-z0-9]+", t) if len(w) > 2 and w not in STOP]
    return [w[:-1] if len(w) > 4 and w.endswith("s") else w for w in ws]        # singular simple

class BM25:
    def __init__(self, docs_tok, k1=1.5, b=0.75):
        self.k1, self.b, self.N = k1, b, len(docs_tok)
        self.dl = np.array([len(d) for d in docs_tok]); self.avg = max(self.dl.mean(), 1)
        df = Counter(); self.inv = defaultdict(list)
        for i, d in enumerate(docs_tok):
            c = Counter(d); df.update(c.keys())
            for w, f in c.items():
                self.inv[w].append((i, f))
        self.idf = {w: np.log(1 + (self.N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
    def scores(self, q):
        s = np.zeros(self.N)
        for w in set(q):
            for i, f in self.inv.get(w, []):
                s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avg))
        return s

class Indice:
    """Colección Chroma + embeddings + BM25 de un conjunto de chunks."""
    def __init__(self, nombre, chunks):
        t0 = time.time()
        self.chunks = chunks
        self.fila = {c["id"]: i for i, c in enumerate(chunks)}
        E = np.array(embed([c["texto"] for c in chunks]), dtype="float32")
        self.E = E / np.linalg.norm(E, axis=1, keepdims=True)
        try: client.delete_collection(nombre)
        except Exception: pass
        self.col = client.create_collection(nombre, metadata={"hnsw:space": "cosine"})
        for i in range(0, len(chunks), 500):
            b = slice(i, i + 500)
            self.col.add(ids=[c["id"] for c in chunks[b]], embeddings=self.E[b].tolist(),
                         documents=[c["texto"] for c in chunks[b]],
                         metadatas=[{"doc": c["doc"], "art": c["art"]} for c in chunks[b]])
        self.bm25 = BM25([tok(c["texto"]) for c in chunks])
        self.n_por_doc = Counter(c["doc"] for c in chunks)
        print(f"Indexados {len(chunks)} chunks en {time.time()-t0:.0f}s")

IDX = Indice("base_principal", chunks)

Indexados 735 chunks en 33s


## 5. Motor RAG
**Recuperación** (`modo`):
- `vector`: top‑k global por similitud coseno (línea base).
- `balanceado`: top‑`por_doc` de **cada documento** y luego los k mejores (evita que un documento largo acapare el contexto en preguntas que cruzan normas).
- `hibrido` (por defecto): candidatos balanceados + BM25, fusionados con **Reciprocal Rank Fusion**.

**Abstención por umbral:** si la distancia del mejor chunk (`dist_top1`) supera `UMBRAL_DIST`, responde *“No encuentro…”* sin llamar al LLM. El umbral se calibra con los datos en la sección 8.

In [8]:
_exp_cache = {}
def expandir_consulta(pregunta):
    """Reescribe la pregunta como frase de texto legal ("se presume inocente") para cerrar brechas léxicas.
    Se concatena a la pregunta original. Una llamada al LLM por pregunta (con caché)."""
    if pregunta not in _exp_cache:
        _exp_cache[pregunta] = ollama.chat(model=LLM_MODEL, options={"temperature": 0, "seed": 42, "num_ctx": 2048, "num_predict": 60}, messages=[
            {"role": "user", "content": "Reescribe esta pregunta como una frase breve con el vocabulario formal con que aparecería en el texto de una ley o "
                                        f"de la Constitución colombiana (sin responderla). Solo la frase.\nPregunta: {pregunta}"}])["message"]["content"].strip()
    return _exp_cache[pregunta]

def recuperar(idx, pregunta, k=4, modo="hibrido", por_doc=2, n_bm25=10, expandir=False):
    """Devuelve (chunks recuperados, distancia del mejor chunk vectorial)."""
    if expandir:
        pregunta = pregunta + " " + expandir_consulta(pregunta)
    q = np.array(embed([pregunta])[0], dtype="float32"); q /= np.linalg.norm(q)
    def dist(cid): return float(1 - idx.E[idx.fila[cid]] @ q)
    if modo == "vector":
        r = idx.col.query(query_embeddings=[q.tolist()], n_results=k)
        ids = r["ids"][0]; d1 = r["distances"][0][0]
    else:
        pool = []
        for d, n in idx.n_por_doc.items():
            r = idx.col.query(query_embeddings=[q.tolist()], n_results=min(por_doc, n), where={"doc": d})
            pool += list(zip(r["ids"][0], r["distances"][0]))
        pool.sort(key=lambda x: x[1]); d1 = pool[0][1]
        if modo == "balanceado":
            ids = [i for i, _ in pool[:k]]
        else:                                                      # híbrido: RRF entre ranking vectorial y BM25
            rank_v = [i for i, _ in pool]
            s = idx.bm25.scores(tok(pregunta))
            rank_b = [idx.chunks[j]["id"] for j in np.argsort(-s)[:n_bm25] if s[j] > 0]
            rrf = defaultdict(float)
            for rank in (rank_v, rank_b):
                for pos, cid in enumerate(rank):
                    rrf[cid] += 1 / (60 + pos + 1)
            ids = [cid for cid, _ in sorted(rrf.items(), key=lambda x: -x[1])[:k]]
    res = [{"id": i, "texto": idx.chunks[idx.fila[i]]["texto"], "doc": idx.chunks[idx.fila[i]]["doc"],
            "art": idx.chunks[idx.fila[i]]["art"], "dist": dist(i)} for i in ids]
    return res, d1

NO_INFO = "No encuentro esa información en los documentos."
SYSTEM = ("Eres un asistente jurídico que responde en español de forma clara y breve. Usa SOLO el contexto entregado; "
          "no uses conocimiento externo ni cites otras leyes o países. Cuando afirmes algo, indica la fuente con el formato "
          "[documento, Art. N], por ejemplo: 'La tutela protege derechos fundamentales [Constitución_2024.pdf, Art. 86].' "
          f"Si el contexto no contiene la respuesta, responde exactamente: '{NO_INFO}'")
OPC = {"temperature": 0, "seed": 42, "num_ctx": 3072}              # determinista; 3072 tokens bastan para k<=6 y caben enteros en la GPU (6 GB)
UMBRAL_DIST = 0.52                                                  # por encima del máx. de las preguntas CON respuesta (≈0.50); se calibra en 8.1

def responder_rag(idx, pregunta, k=4, modo="hibrido", usar_umbral=True):
    t0 = time.time()
    ctx, d1 = recuperar(idx, pregunta, k, modo)
    if usar_umbral and UMBRAL_DIST and d1 > UMBRAL_DIST:
        return dict(resp=NO_INFO, raw=NO_INFO, ctx=ctx, seg=time.time() - t0, dist_top1=d1, abstuvo=True)
    contexto = "\n\n".join(f"[{c['doc']}, Art. {c['art']}]\n{c['texto']}" for c in ctx)
    opc = {**OPC, "num_ctx": 4096 if k > 6 else OPC["num_ctx"]}
    raw = ollama.chat(model=LLM_MODEL, options=opc, messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}])["message"]["content"]
    # Fuentes = solo los chunks que el modelo citó (si no citó ninguno, los 2 mejores), no todos los recuperados
    citados = set(re.findall(r"Art\.\s*(\d+)", raw))
    usados = [c for c in ctx if citados & set(re.findall(r"\d+", c["art"]))] or ctx[:2]
    fuentes = "; ".join(sorted({f"[{c['doc']}, Art. {c['art']}]" for c in usados}))
    resp = raw if raw.strip().lower().startswith("no encuentro") else f"{raw}\n\n**Fuentes:** {fuentes}"
    return dict(resp=resp, raw=raw, ctx=ctx, seg=time.time() - t0, dist_top1=d1, abstuvo=False)

def responder_sin_rag(pregunta):
    return ollama.chat(model=LLM_MODEL, options=OPC, messages=[
        {"role": "system", "content": "Eres un asistente jurídico que responde en español de forma breve."},
        {"role": "user", "content": pregunta}])["message"]["content"]

r = responder_rag(IDX, "¿Qué es la acción de tutela?")
print(r["resp"], f"\n\n({r['seg']:.1f}s, dist_top1={r['dist_top1']:.3f})")

La acción de tutela es un procedimiento jurídico que permite a una persona reclamar la protección inmediata de sus derechos constitucionales fundamentales ante los jueces, en todo momento y lugar, cuando estos resulten vulnerados o amenazados por la acción o la omisión de cualquier autoridad pública.

**Fuentes:** [Constitución_2024.pdf, Art. 380]; [Constitución_2024.pdf, Art. 86] 

(11.1s, dist_top1=0.349)


## 6. Conjunto de evaluación
Cada pregunta trae la **respuesta de referencia** (`ref`), `tipo` y:
- `gold`: lista de pares `(documento, regex)`. Un chunk es **relevante** si pertenece a ese documento y **contiene una frase distintiva** (una sola regex por par, para que el tamaño del chunk no cambie qué cuenta como relevante). Las preguntas **cruzadas** tienen dos pares (uno por documento).
- `clave`: patrones que debe contener una respuesta correcta (p. ej. *quince|15*): mide **datos correctos**, algo que la similitud de embeddings no detecta.
- `gold=None`: la pregunta **no tiene respuesta** en los documentos (se espera abstención). Hay negativas *lejanas* y *cercanas* (mismo dominio, pero la respuesta no está).

In [9]:
C_, L55, L81 = "constituci", "1755", "1581"
EVAL = [
 # ---------------- Constitución ----------------
 dict(tipo="simple", q="¿Qué es la acción de tutela y ante quién se presenta?", gold=[(C_, r"Toda persona tendr[aá] acci[oó]n de tutela")],
      clave=[r"preferente y sumario|protecci[oó]n inmediata|jueces"],
      ref="Toda persona puede reclamar ante los jueces, en todo momento y lugar, mediante un procedimiento preferente y sumario, la protección inmediata de sus derechos constitucionales fundamentales (art. 86)."),
 dict(tipo="simple", q="¿Qué dice la Constitución sobre el derecho de petición?", gold=[(C_, r"peticiones respetuosas")], clave=[r"respetuosas"],
      ref="Toda persona tiene derecho a presentar peticiones respetuosas a las autoridades por motivos de interés general o particular y a obtener pronta resolución (art. 23)."),
 dict(tipo="simple", q="¿Hay pena de muerte en Colombia?", gold=[(C_, r"No habr[aá] pena de muerte")], clave=[r"no habr[aá] pena de muerte|art[ií]culo 11|inviolable"],
      ref="No. El derecho a la vida es inviolable y no habrá pena de muerte (art. 11)."),
 dict(tipo="simple", q="¿Cómo define la Constitución a Colombia en su primer artículo?", gold=[(C_, r"Estado social de derecho, organizado en forma de Rep[uú]blica unitaria")],
      clave=[r"social de derecho"],
      ref="Colombia es un Estado social de derecho, organizado en forma de República unitaria, descentralizada, con autonomía de sus entidades territoriales, democrática, participativa y pluralista."),
 dict(tipo="simple", q="¿Qué dice la Constitución sobre la igualdad ante la ley?", gold=[(C_, r"nacen libres e iguales")], clave=[r"libres e iguales|igualdad"],
      ref="Todas las personas nacen libres e iguales ante la ley, reciben la misma protección y trato de las autoridades y gozan de los mismos derechos, libertades y oportunidades sin discriminación (art. 13)."),
 dict(tipo="simple", q="¿Por cuántos años es elegido el Presidente de la República?", gold=[(C_, r"Presidente de la Rep[uú]blica ser[aá] elegido para un per[ií]odo de cuatro a[nñ]os")],
      clave=[r"cuatro|\b4\b"], ref="El Presidente de la República es elegido para un período de cuatro años (art. 190)."),
 dict(tipo="simple", q="¿Qué dice la Constitución sobre la presunción de inocencia?", gold=[(C_, r"se presume inocente")], clave=[r"inocente"],
      ref="Toda persona se presume inocente mientras no se la haya declarado judicialmente culpable (art. 29)."),
 dict(tipo="simple", q="¿Qué dice la Constitución sobre el derecho a un ambiente sano?", gold=[(C_, r"derecho a gozar de un ambiente sano")], clave=[r"ambiente sano"],
      ref="Todas las personas tienen derecho a gozar de un ambiente sano y la ley garantizará la participación de la comunidad en las decisiones que puedan afectarlo (art. 79)."),
 dict(tipo="simple", q="¿Qué derecho tienen las personas sobre la información que se recoge sobre ellas en bancos de datos?", gold=[(C_, r"conocer, actualizar y rectificar")],
      clave=[r"actualizar|rectificar|conocer"],
      ref="Tienen derecho a conocer, actualizar y rectificar las informaciones que se hayan recogido sobre ellas en bancos de datos y archivos de entidades públicas y privadas (art. 15)."),
 # ---------------- Ley 1755 de 2015 (derecho de petición) ----------------
 dict(tipo="simple", q="¿En cuántos días se debe resolver una petición en general?", gold=[(L55, r"quince \(15\) d[ií]as siguientes")], clave=[r"quince|\b15\b"],
      ref="Salvo norma legal especial, toda petición debe resolverse dentro de los quince (15) días siguientes a su recepción."),
 dict(tipo="simple", q="¿En cuántos días se resuelven las peticiones de documentos y de información?", gold=[(L55, r"diez \(10\) d[ií]as siguientes")], clave=[r"diez|\b10\b"],
      ref="Las peticiones de documentos y de información deben resolverse dentro de los diez (10) días siguientes a su recepción."),
 dict(tipo="simple", q="¿En cuántos días se resuelven las consultas a las autoridades?", gold=[(L55, r"treinta \(30\) d[ií]as siguientes")], clave=[r"treinta|\b30\b"],
      ref="Las peticiones en que se eleva una consulta a las autoridades deben resolverse dentro de los treinta (30) días siguientes a su recepción."),
 dict(tipo="simple", q="¿Qué es el desistimiento tácito en una petición incompleta?", gold=[(L55, r"desistimiento t[aá]cito")], clave=[r"desisti|archiv"],
      ref="Si el peticionario no completa la información o los documentos requeridos dentro del término legal, se entiende que desistió de su petición y se archiva el expediente."),
 dict(tipo="simple", q="¿Qué peticiones tienen atención prioritaria?", gold=[(L55, r"atenci[oó]n prioritaria")], clave=[r"periodista|perjuicio irremediable|prioritari"],
      ref="Las peticiones de reconocimiento de un derecho fundamental cuando deba reconocerse para evitar un perjuicio irremediable al peticionario, y las de periodistas, entre otras, tienen atención prioritaria."),
 # ---------------- Ley 1581 de 2012 (protección de datos personales) ----------------
 dict(tipo="simple", q="¿Cuál es el objeto de la ley de protección de datos personales?", gold=[(L81, r"bases de datos o archivos")], clave=[r"conocer|actualizar|rectificar"],
      ref="Desarrollar el derecho constitucional de todas las personas a conocer, actualizar y rectificar las informaciones recogidas sobre ellas en bases de datos o archivos, y los demás derechos y garantías del artículo 15 de la Constitución."),
 dict(tipo="simple", q="¿Qué entidad ejerce la vigilancia de la protección de datos personales?", gold=[(L81, r"Delegatura para la Protecci[oó]n de Datos Personales")],
      clave=[r"Superintendencia de Industria y Comercio|\bSIC\b"],
      ref="La Superintendencia de Industria y Comercio, a través de la Delegatura para la Protección de Datos Personales."),
 dict(tipo="simple", q="¿Qué es un dato sensible?", gold=[(L81, r"afectan la intimidad del Titular")], clave=[r"intimidad|discrimina"],
      ref="Aquel que afecta la intimidad del Titular o cuyo uso indebido puede generar su discriminación, como el origen racial o étnico, la orientación política, las convicciones religiosas, la salud o la vida sexual."),
 dict(tipo="simple", q="¿Qué tipo de consentimiento se requiere para tratar datos personales?", gold=[(L81, r"previo, expreso e informado")], clave=[r"previo", r"expreso", r"informado"],
      ref="El tratamiento solo puede ejercerse con el consentimiento previo, expreso e informado del Titular."),
 dict(tipo="simple", q="¿Cuál es el término máximo para atender un reclamo sobre datos personales?", gold=[(L81, r"quince \(15\) d[ií]as h[aá]biles")], clave=[r"quince|\b15\b"],
      ref="El término máximo para atender el reclamo es de quince (15) días hábiles contados desde el día siguiente a su recibo."),
 dict(tipo="simple", q="¿En cuántos días se atiende una consulta sobre datos personales?", gold=[(L81, r"diez \(10\) d[ií]as h[aá]biles")], clave=[r"diez|\b10\b"],
      ref="La consulta será atendida en un término máximo de diez (10) días hábiles contados desde la fecha de su recibo."),
 # ---------------- Cruzadas (necesitan dos documentos) ----------------
 dict(tipo="cruzada", q="¿Qué artículo de la Constitución consagra el derecho de petición y en cuántos días debe resolverse una petición en general según la ley?",
      gold=[(C_, r"peticiones respetuosas"), (L55, r"quince \(15\) d[ií]as siguientes")], clave=[r"\b23\b", r"quince|\b15\b"],
      ref="El derecho de petición está en el artículo 23 de la Constitución, y según la Ley 1755 toda petición debe resolverse en quince (15) días."),
 dict(tipo="cruzada", q="¿Qué derecho constitucional protege el conocimiento y la rectificación de los datos personales y qué autoridad vigila su tratamiento según la ley?",
      gold=[(C_, r"conocer, actualizar y rectificar"), (L81, r"Delegatura para la Protecci[oó]n de Datos Personales")],
      clave=[r"actualizar|rectificar|\b15\b", r"Superintendencia de Industria y Comercio|\bSIC\b"],
      ref="El artículo 15 de la Constitución reconoce el derecho a conocer, actualizar y rectificar la información; la Ley 1581 asigna la vigilancia a la Superintendencia de Industria y Comercio."),
 dict(tipo="cruzada", q="¿Qué consentimiento exige la ley para tratar datos personales y qué protege la Constitución sobre la intimidad de las personas?",
      gold=[(L81, r"previo, expreso e informado"), (C_, r"derecho a su intimidad personal y familiar y a su buen nombre")],
      clave=[r"previo|expreso|informado", r"intimidad|buen nombre"],
      ref="La Ley 1581 exige consentimiento previo, expreso e informado; la Constitución protege la intimidad personal y familiar y el buen nombre (art. 15)."),
 # ---------------- Sin respuesta en los documentos (gold=None) ----------------
 dict(tipo="negativa_lejana", q="¿Cuál es la receta tradicional de la bandeja paisa?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Qué sanción específica recibió una empresa en 2025 por filtrar datos personales?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Qué dice el artículo 300 de la Ley 1581 de 2012?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Qué establece la Ley 1581 de 2012 sobre el uso de cookies en los sitios web?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Qué dice el artículo 900 de la Constitución Política de Colombia?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Quién es el actual Superintendente de Industria y Comercio?", gold=None, clave=None, ref=NO_INFO),
 dict(tipo="negativa_cercana", q="¿Qué sentencia de la Corte Constitucional declaró exequible la Ley 1581 de 2012?", gold=None, clave=None, ref=NO_INFO),
]
# Solo se evalúan las preguntas cuyos documentos están cargados
nombres_docs = [d.lower() for d in docs]
EVAL = [e for e in EVAL if e["gold"] is None or all(any(sub in n for n in nombres_docs) for sub, _ in e["gold"])]
print(len(EVAL), "preguntas |", pd.Series([e["tipo"] for e in EVAL]).value_counts().to_dict())

30 preguntas | {'simple': 20, 'negativa_cercana': 6, 'cruzada': 3, 'negativa_lejana': 1}


## 7. Métricas de recuperación
- **Hit@k:** ¿hay algún chunk relevante entre los k primeros? · **MRR:** 1/posición del primero.
- **Recall@k** = relevantes recuperados / min(k, nº de relevantes) (así el máximo posible es 1 aunque haya muchos chunks relevantes).
- **Cobertura@k** (cruzadas): fracción de documentos requeridos que aparecen en el top‑k.
- Las preguntas con **0 chunks relevantes** no se cuentan

In [10]:
def norm(t): return re.sub(r"\s+", " ", t)

def relevantes(chunks, gold):
    """Un conjunto de ids relevantes por cada par (documento, regex)."""
    return [{c["id"] for c in chunks if sub in c["doc"].lower() and re.search(rx, norm(c["texto"]), re.I)} for sub, rx in gold]

# --- Validación del conjunto de evaluación ---
for e in EVAL:
    if e["gold"] is None: continue
    n = [len(s) for s in relevantes(chunks, e["gold"])]
    if min(n) == 0:
        print("⚠️  gold sin coincidencias →", e["q"], "| chunks por par:", n, "| revisa la regex:", [rx for _, rx in e["gold"]])
print("Validación terminada.")

def eval_recuperacion(idx, ks=(1, 3, 5), modo="hibrido", verbose=False):
    kmax = max(ks); filas, omitidas = [], []
    for e in EVAL:
        if e["gold"] is None: continue
        sets = relevantes(idx.chunks, e["gold"]); rel = set().union(*sets)
        if min(len(s) for s in sets) == 0:
            omitidas.append(e["q"]); continue
        top, d1 = recuperar(idx, e["q"], kmax, modo); ids = [r["id"] for r in top]
        pos = next((i + 1 for i, cid in enumerate(ids) if cid in rel), None)
        f = {"tipo": e["tipo"], "q": e["q"], "n_rel": len(rel), "mrr": 1 / pos if pos else 0.0, "dist_top1": d1}
        for k in ks:
            f[f"hit@{k}"] = int(pos is not None and pos <= k)
            f[f"recall@{k}"] = len(rel & set(ids[:k])) / min(k, len(rel))
            f[f"cov@{k}"] = float(np.mean([bool(s & set(ids[:k])) for s in sets]))
        filas.append(f)
    if omitidas and verbose:
        print("Omitidas (sin chunks relevantes):", omitidas)
    return pd.DataFrame(filas)

df_ret = eval_recuperacion(IDX, verbose=True)
df_ret.to_csv(SALIDAS / "rag_recuperacion.csv", index=False)
display(df_ret.drop(columns=["recall@1", "recall@3", "cov@1", "cov@3"]))
print("\nPromedio global:\n", df_ret.drop(columns=["q", "tipo"]).mean().round(3).to_string())
print("\nPor tipo de pregunta:\n", df_ret.drop(columns=["q"]).groupby("tipo").mean().round(3)[["mrr", "hit@1", "hit@3", "hit@5", "cov@5"]])

Validación terminada.


,tipo,q,n_rel,mrr,dist_top1,hit@1,hit@3,hit@5,recall@5,cov@5
0,simple,¿Qué es la acción de tutela y ante quién se pr...,1,1.000000,0.305600,1,1,1,1.000000,1.0
1,simple,¿Qué dice la Constitución sobre el derecho de ...,1,0.250000,0.333935,0,0,1,1.000000,1.0
2,simple,¿Hay pena de muerte en Colombia?,1,0.500000,0.455190,0,1,1,1.000000,1.0
3,simple,¿Cómo define la Constitución a Colombia en su ...,1,0.500000,0.363885,0,1,1,1.000000,1.0
4,simple,¿Qué dice la Constitución sobre la igualdad an...,1,0.000000,0.444561,0,0,0,0.000000,0.0
5,simple,¿Por cuántos años es elegido el Presidente de ...,1,1.000000,0.242700,1,1,1,1.000000,1.0
6,simple,¿Qué dice la Constitución sobre la presunción ...,1,0.000000,0.495900,0,0,0,0.000000,0.0
7,simple,¿Qué dice la Constitución sobre el derecho a u...,1,1.000000,0.328070,1,1,1,1.000000,1.0
8,simple,¿Qué derecho tienen las personas sobre la info...,1,1.000000,0.304248,1,1,1,1.000000,1.0
9,simple,¿En cuántos días se debe resolver una petición...,1,1.000000,0.284582,1,1,1,1.000000,1.0



Promedio global:
 n_rel        1.522
mrr          0.808
dist_top1    0.336
hit@1        0.739
recall@1     0.739
cov@1        0.674
hit@3        0.870
recall@3     0.768
cov@3        0.848
hit@5        0.913
recall@5     0.806
cov@5        0.891

Por tipo de pregunta:
            mrr  hit@1  hit@3  hit@5  cov@5
tipo                                      
cruzada  1.000    1.0   1.00    1.0  0.833
simple   0.779    0.7   0.85    0.9  0.900


## 8. Métricas de generación: RAG vs sin RAG

1. **Dato clave:** ¿la respuesta contiene los datos correctos (p. ej. *quince|15*)? Detecta errores que la similitud no ve.
2. **Similitud semántica** (coseno) con la respuesta de referencia.
3. **Fidelidad** (juez LLM, 1–5). El juez es el mismo modelo de 3B: úsalo como apoyo, no como verdad; la referencia es tu **rúbrica manual**.
4. **Fuente:** `cita_modelo` (¿el modelo citó artículo?) y `ctx_relevante` (¿el contexto recuperado contenía el chunk correcto?). Las fuentes finales las añade el código.
5. **Abstención:** en preguntas sin respuesta, ¿rechazó correctamente?

In [11]:
MAX_PREGUNTAS = None     # None = todas
JUZGAR = True            # juez LLM de fidelidad

def coseno(a, b):
    a, b = np.array(a), np.array(b); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def juez_fidelidad(pregunta, contexto, respuesta):
    p = f"""Evalúa si la RESPUESTA se apoya únicamente en el CONTEXTO (sin inventar datos).
Puntaje 1 a 5: 5 = totalmente respaldada; 3 = parcialmente; 1 = contradice o inventa.
Devuelve SOLO JSON: {{"puntaje": <1-5>}}
PREGUNTA: {pregunta}
CONTEXTO: {contexto[:3000]}
RESPUESTA: {respuesta}"""
    r = ollama.chat(model=LLM_MODEL, options={"temperature": 0, "seed": 42, "num_ctx": 3072}, format="json",
                    messages=[{"role": "user", "content": p}])
    try: return int(json.loads(r["message"]["content"])["puntaje"])
    except Exception: return None

def dato_clave(resp, clave):
    """Fracción de patrones clave presentes en la respuesta (NaN si la pregunta no tiene clave)."""
    return float(np.mean([bool(re.search(k, resp, re.I)) for k in clave])) if clave else np.nan

def eval_generacion(idx, k=4, modo="hibrido", usar_umbral=True, preguntas=None):
    filas = []
    for e in (preguntas or (EVAL[:MAX_PREGUNTAS] if MAX_PREGUNTAS else EVAL)):
        R = responder_rag(idx, e["q"], k, modo, usar_umbral)
        base = responder_sin_rag(e["q"])
        ref_emb = embed([e["ref"]])[0]
        contexto = "\n".join(c["texto"] for c in R["ctx"])
        rel = set().union(*relevantes(idx.chunks, e["gold"])) if e["gold"] else set()
        no_info = R["abstuvo"] or R["raw"].strip().lower().startswith("no encuentro")      # se abstuvo (umbral) o el LLM dijo "no encuentro"
        filas.append({
            "tipo": e["tipo"], "q": e["q"], "resp_rag": R["resp"], "resp_sin_rag": base,
            "dist_top1": R["dist_top1"], "abstuvo": R["abstuvo"], "abstencion_indebida": bool(no_info and e["gold"]), "lat_rag_s": R["seg"],
            "dato_rag": (0.0 if no_info else dato_clave(R["raw"], e["clave"])) if e["clave"] else np.nan,   # abstenerse NO cuenta como acierto
            "dato_sin_rag": dato_clave(base, e["clave"]),
            "sim_rag": coseno(embed([R["raw"]])[0], ref_emb), "sim_sin_rag": coseno(embed([base])[0], ref_emb),
            "fidelidad_rag": juez_fidelidad(e["q"], contexto, R["raw"]) if (JUZGAR and not R["abstuvo"]) else None,
            "fidelidad_sin_rag": juez_fidelidad(e["q"], contexto, base) if JUZGAR else None,
            "cita_modelo": bool(re.search(r"(?i)art[ií]culo\s+\d+|\[[^\]]*Art", R["raw"])),
            "ctx_relevante": bool(rel & {c["id"] for c in R["ctx"]}) if e["gold"] else None,
            "rechaza_correcto": no_info if e["gold"] is None else None,
        })
    return pd.DataFrame(filas)

df_gen = eval_generacion(IDX)
df_gen.to_csv(SALIDAS / "rag_generacion.csv", index=False)
con = df_gen[df_gen["rechaza_correcto"].isna()]
sin = df_gen[df_gen["rechaza_correcto"].notna()]
print("PREGUNTAS CON RESPUESTA (n =", len(con), ")")
print(con[["dato_rag", "dato_sin_rag", "sim_rag", "sim_sin_rag", "fidelidad_rag", "fidelidad_sin_rag", "lat_rag_s"]].astype(float).mean().round(3).to_string())
print(f"cita_modelo: {100*con['cita_modelo'].mean():.0f} % | contexto con el chunk correcto: {100*con['ctx_relevante'].astype(float).mean():.0f} %")
ind = con[con["abstencion_indebida"]]
print(f"⚠️ Abstenciones indebidas (tenía respuesta y dijo 'no encuentro'): {len(ind)} de {len(con)}" + (" → " + "; ".join(f"{q[:45]}… (dist {d:.3f})" for q, d in zip(ind['q'], ind['dist_top1'])) if len(ind) else ""))
print("\nPOR TIPO (dato clave RAG vs sin RAG):\n", con.groupby("tipo")[["dato_rag", "dato_sin_rag"]].mean().round(2))
print("\nPREGUNTAS SIN RESPUESTA (n =", len(sin), ")\n", sin.groupby("tipo")["rechaza_correcto"].agg(["sum", "count"]).rename(columns={"sum": "rechazó", "count": "total"}))

PREGUNTAS CON RESPUESTA (n = 23 )
dato_rag             0.891
dato_sin_rag         0.319
sim_rag              0.796
sim_sin_rag          0.699
fidelidad_rag        3.348
fidelidad_sin_rag    1.957
lat_rag_s            3.378
cita_modelo: 78 % | contexto con el chunk correcto: 91 %
⚠️ Abstenciones indebidas (tenía respuesta y dijo 'no encuentro'): 2 de 23 → ¿Qué dice la Constitución sobre la igualdad a… (dist 0.445); ¿Qué dice la Constitución sobre la presunción… (dist 0.496)

POR TIPO (dato clave RAG vs sin RAG):
          dato_rag  dato_sin_rag
tipo                           
cruzada      0.83          0.33
simple       0.90          0.32

PREGUNTAS SIN RESPUESTA (n = 7 )
                  rechazó  total
tipo                           
negativa_cercana       4      6
negativa_lejana     True      1


### 8.1 Calibración del umbral de abstención
Las preguntas con respuesta deben tener una `dist_top1` baja y las que no la tienen, alta. El umbral se elige **con tus datos**, no a ojo.

In [12]:
d_ok = df_gen[df_gen["rechaza_correcto"].isna()]["dist_top1"]
d_no = df_gen[df_gen["rechaza_correcto"].notna()]["dist_top1"]
print(f"dist_top1 con respuesta: mediana {d_ok.median():.3f}, máx {d_ok.max():.3f}")
print(f"dist_top1 sin respuesta: mediana {d_no.median():.3f}, mín {d_no.min():.3f}")
if d_ok.max() < d_no.min():
    print(f"→ separables. Umbral sugerido: {(d_ok.max() + d_no.min()) / 2:.3f}  (asígnalo a UMBRAL_DIST y repite la evaluación)")
else:
    print("→ se solapan: sube el umbral por encima del máximo con respuesta y acepta que alguna negativa cercana llegue al LLM; "
          "el prompt le pide abstenerse en ese caso.")
print("UMBRAL_DIST actual:", UMBRAL_DIST)

dist_top1 con respuesta: mediana 0.320, máx 0.496
dist_top1 sin respuesta: mediana 0.471, mín 0.412
→ se solapan: sube el umbral por encima del máximo con respuesta y acepta que alguna negativa cercana llegue al LLM; el prompt le pide abstenerse en ese caso.
UMBRAL_DIST actual: 0.52


### 8.2 ¿Se abstiene el LLM por sí mismo? (negativas **sin** umbral)
Con el umbral activo, las negativas lejanas se rechazan sin llamar al LLM, lo que infla la métrica. Aquí se **apaga** el umbral para medir si el modelo, viendo contexto irrelevante, responde “No encuentro…” o alucina.

In [13]:
negativas = [e for e in EVAL if e["gold"] is None]
df_neg = eval_generacion(IDX, usar_umbral=False, preguntas=negativas)
res_neg = df_neg.groupby("tipo")["rechaza_correcto"].agg(rechazó="sum", total="count")
display(res_neg)
for _, r in df_neg[~df_neg["rechaza_correcto"].astype(bool)].iterrows():
    print("ALUCINÓ →", r["q"], "\n   ", r["resp_rag"][:250].replace("\n", " "))

,rechazó,total
tipo,,
negativa_cercana,4,6
negativa_lejana,1,1


ALUCINÓ → ¿Qué dice el artículo 300 de la Ley 1581 de 2012? 
    El artículo 300 de la Ley 1581 de 2012 establece que corresponde a las asambleas departamentales, por medio de ordenanzas, reglamentar el ejercicio de las funciones y la prestación de los servicios a cargo del departamento, entre otros aspectos.  **F
ALUCINÓ → ¿Qué sentencia de la Corte Constitucional declaró exequible la Ley 1581 de 2012? 
    Según la información proporcionada, la Corte Constitucional declaró exequible el Acto Legislativo 2 de 2004, salvo el inciso tercero del parágrafo transitorio del artículo 4º, que fue declarado inexequible. Sin embargo, no se menciona la sentencia es


### 8.3 Rúbrica manual (la métrica más confiable)
Se exporta un CSV con las respuestas lado a lado. Ábrelo en Excel, califica de **1 a 5** (exactitud + uso correcto de la fuente) en `puntaje_rag` y `puntaje_sin_rag`, guárdalo y vuelve a ejecutar esta celda para ver los promedios.

In [14]:
RUBRICA = SALIDAS / "rag_rubrica_manual.csv"
if not RUBRICA.exists():
    pd.DataFrame({"tipo": df_gen["tipo"], "pregunta": df_gen["q"], "resp_rag": df_gen["resp_rag"],
                  "resp_sin_rag": df_gen["resp_sin_rag"], "puntaje_rag": "", "puntaje_sin_rag": ""}).to_csv(RUBRICA, index=False, encoding="utf-8-sig")
    print("Rúbrica creada en", RUBRICA, "→ califícala y vuelve a ejecutar esta celda.")
else:
    rub = pd.read_csv(RUBRICA)
    for c in ("puntaje_rag", "puntaje_sin_rag"):
        rub[c] = pd.to_numeric(rub[c], errors="coerce")
    if rub["puntaje_rag"].notna().any():
        print("Promedio manual (1-5):\n", rub[["puntaje_rag", "puntaje_sin_rag"]].mean().round(2).to_string())
        print("\nPor tipo:\n", rub.groupby("tipo")[["puntaje_rag", "puntaje_sin_rag"]].mean().round(2))
    else:
        print("La rúbrica existe pero está vacía:", RUBRICA)

Rúbrica creada en /home/isabellap/Escritorio/Entregable Transformers/salidas/rag_rubrica_manual.csv → califícala y vuelve a ejecutar esta celda.


## 9. Ablaciones
1. **Chunking × tipo de recuperación** (vector / balanceado / híbrido). Con ~20 preguntas, **una pregunta cambia el MRR en ≈ 0.05**: diferencias menores a eso son ruido (además Chroma/HNSW es aproximado y puede variar levemente entre corridas). Elige por criterio, no por el último decimal.
2. **Valor de k en la generación** (calidad de la respuesta con 2, 4 u 8 chunks de contexto).

La latencia incluye el embedding de la pregunta **sin caché** (el caché hacía que antes saliera 0.002 s).

In [15]:
HACER_ABLACION = True
def latencia(idx, modo, n=6):
    qs = [e["q"] for e in EVAL if e["gold"]][:n]; t = []
    for q in qs:
        t0 = time.time(); ollama.embed(model=EMB_MODEL, input=[q]); recuperar(idx, q, 5, modo); t.append(time.time() - t0)
    return float(np.mean(t))

if HACER_ABLACION:
    resultados, indices = [], {}
    for modo_c, size, ov in [("fijo", 400, 60), ("fijo", 800, 120), ("fijo", 1200, 180), ("articulo", 1200, 150)]:
        ch = construir_chunks(docs, modo_c, size, ov)
        ix = IDX if (modo_c, size, ov) == tuple(CFG_CHUNK.values()) else Indice(f"abl_{modo_c}_{size}", ch)
        indices[(modo_c, size)] = ix
        for modo_r in ("vector", "balanceado", "hibrido"):
            d = eval_recuperacion(ix, modo=modo_r)
            resultados.append({"chunking": modo_c, "size": size, "n_chunks": len(ch), "recuperacion": modo_r, "n_preg": len(d),
                               **d[["mrr", "hit@1", "hit@3", "hit@5", "recall@5", "cov@5"]].mean().round(3).to_dict(),
                               "lat_s": round(latencia(ix, modo_r), 3)})
    df_abl = pd.DataFrame(resultados); df_abl.to_csv(SALIDAS / "rag_ablacion.csv", index=False)
    display(df_abl)

Indexados 1607 chunks en 46s
Indexados 826 chunks en 27s
Indexados 548 chunks en 22s


,chunking,size,n_chunks,recuperacion,n_preg,mrr,hit@1,hit@3,hit@5,recall@5,cov@5,lat_s
0,fijo,400,1607,vector,23,0.726,0.696,0.739,0.783,0.729,0.783,0.036
1,fijo,400,1607,balanceado,23,0.743,0.696,0.783,0.826,0.755,0.804,0.077
2,fijo,400,1607,hibrido,23,0.726,0.609,0.826,0.870,0.813,0.848,0.079
3,fijo,800,826,vector,23,0.891,0.826,0.957,0.957,0.835,0.935,0.030
4,fijo,800,826,balanceado,23,0.891,0.826,0.957,0.957,0.773,0.913,0.061
5,fijo,800,826,hibrido,23,0.906,0.870,0.957,0.957,0.809,0.935,0.062
6,fijo,1200,548,vector,23,0.784,0.696,0.870,0.913,0.809,0.891,0.028
7,fijo,1200,548,balanceado,23,0.761,0.696,0.826,0.826,0.691,0.783,0.057
8,fijo,1200,548,hibrido,23,0.826,0.783,0.870,0.870,0.789,0.848,0.060
9,articulo,1200,735,vector,23,0.797,0.739,0.870,0.870,0.793,0.848,0.030


In [16]:
HACER_ABLACION_K = True
if HACER_ABLACION_K:
    sub = [e for e in EVAL if e["gold"] and e["clave"]][::2][:10]          # subconjunto para limitar las llamadas al LLM
    filas = []
    for k in (2, 4, 8):
        d = eval_generacion(IDX, k=k, usar_umbral=False, preguntas=sub)
        filas.append({"k": k, "n_preg": len(d), "dato_rag": d["dato_rag"].mean(), "sim_rag": d["sim_rag"].mean(),
                      "ctx_relevante": d["ctx_relevante"].astype(float).mean(), "lat_s": d["lat_rag_s"].mean()})
    df_k = pd.DataFrame(filas).round(3); df_k.to_csv(SALIDAS / "rag_ablacion_k.csv", index=False)
    display(df_k)

,k,n_preg,dato_rag,sim_rag,ctx_relevante,lat_s
0,2,10,0.7,0.699,0.7,2.772
1,4,10,0.8,0.719,0.8,2.831
2,8,10,0.9,0.774,0.9,11.116


### 9.3 Expansión de consulta (análisis de errores de recuperación)
Algunas preguntas fallan por **brecha léxica**: el usuario dice *“presunción de inocencia”* y el artículo dice *“se presume inocente”*; ni BM25 ni el embedding lo cierran. Se prueba reescribir la pregunta con el LLM en vocabulario legal y concatenarla (`expandir=True`).

In [17]:
HACER_EXPANSION = True
if HACER_EXPANSION:
    filas = []
    for exp in (False, True):
        rs = []
        for e in EVAL:
            if e["gold"] is None: continue
            sets = relevantes(IDX.chunks, e["gold"]); rel = set().union(*sets)
            if min(len(s) for s in sets) == 0: continue
            top, _ = recuperar(IDX, e["q"], 5, "hibrido", expandir=exp); ids = [r["id"] for r in top]
            pos = next((i + 1 for i, cid in enumerate(ids) if cid in rel), None)
            rs.append(dict(q=e["q"], mrr=1 / pos if pos else 0.0, hit5=int(pos is not None)))
        d = pd.DataFrame(rs); filas.append(dict(expandir=exp, n=len(d), mrr=round(d.mrr.mean(), 3), hit5=round(d.hit5.mean(), 3)))
        if not exp: fallidas = d[d.hit5 == 0].q.tolist()
    display(pd.DataFrame(filas)); print("Preguntas sin chunk relevante en el top-5 (sin expansión):", fallidas)

,expandir,n,mrr,hit5
0,False,23,0.808,0.913
1,True,23,0.812,1.000


Preguntas sin chunk relevante en el top-5 (sin expansión): ['¿Qué dice la Constitución sobre la igualdad ante la ley?', '¿Qué dice la Constitución sobre la presunción de inocencia?']


## 10. Interfaz Gradio

In [18]:
import gradio as gr
if "demo" in globals():
    try: demo.close()          # libera el puerto 7860 de una ejecución anterior
    except Exception: pass
IDX_FINAL = IDX        # reemplaza por la mejor configuración de la ablación, p. ej. indices[("fijo", 400)]

def app(pregunta, k, modo, usar_rag):
    if usar_rag:
        R = responder_rag(IDX_FINAL, pregunta, int(k), modo)
        fuentes = "\n\n".join(f"**{c['doc']} · Art. {c['art']}** (distancia {c['dist']:.3f})\n\n{c['texto'][:300]}…" for c in R["ctx"])
        return R["resp"], f"*Distancia del mejor fragmento: {R['dist_top1']:.3f} (umbral de abstención {UMBRAL_DIST})*\n\n" + fuentes
    return responder_sin_rag(pregunta), "—"

args = dict(fn=app,
    inputs=[gr.Textbox(label="Pregunta"), gr.Slider(1, 8, value=4, step=1, label="k (chunks recuperados)"),
            gr.Radio(["hibrido", "balanceado", "vector"], value="hibrido", label="Recuperación"), gr.Checkbox(value=True, label="Usar RAG")],
    outputs=[gr.Markdown(label="Respuesta"), gr.Markdown(label="Fuentes recuperadas")],
    title="⚖️ Asistente RAG — Constitución y normas colombianas (Ollama)")
try:
    demo = gr.Interface(flagging_mode="never", **args)        # Gradio ≥ 5
except TypeError:
    demo = gr.Interface(allow_flagging="never", **args)       # Gradio 4
demo.launch(inbrowser=not EN_COLAB, share=EN_COLAB)

/home/isabellap/Escritorio/Entregable Transformers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


## 11. Conclusiones

In [19]:
def borrador_rag():
    print("BORRADOR DE CONCLUSIONES — RAG\n")
    g = df_ret.drop(columns=["q", "tipo"]).mean()
    print(f"1) RECUPERACIÓN (modo híbrido, chunk por artículo, {len(df_ret)} preguntas evaluables): Hit@1 {g['hit@1']:.2f}, Hit@3 {g['hit@3']:.2f}, "
          f"Hit@5 {g['hit@5']:.2f}, MRR {g['mrr']:.2f}, Recall@5 {g['recall@5']:.2f}.")
    cr = df_ret[df_ret.tipo == "cruzada"]
    if len(cr): print(f"   - Preguntas cruzadas: cobertura@5 = {cr['cov@5'].mean():.2f} (fracción de documentos requeridos presentes en el contexto).")
    if "df_abl" in globals():
        a = df_abl.sort_values("mrr", ascending=False).iloc[0]
        p = df_abl[(df_abl.chunking == "articulo") & (df_abl.recuperacion == "hibrido")].iloc[0]
        un = 1 / max(int(p["n_preg"]), 1)
        print(f"   - Ablación: mejor MRR {a['mrr']:.2f} ({a['chunking']}/{a['size']}/{a['recuperacion']}); la configuración elegida (por artículo, híbrido) da {p['mrr']:.2f}. "
              f"Con {int(p['n_preg'])} preguntas, 1 pregunta ≈ {un:.2f} de MRR: la diferencia es ruido, y el chunk por artículo se elige porque da cita exacta (los chunks fijos pueden etiquetar mal el artículo).")
    c = df_gen[df_gen["rechaza_correcto"].isna()]; s = df_gen[df_gen["rechaza_correcto"].notna()]
    print(f"\n2) GENERACIÓN (n = {len(c)} con respuesta): dato clave RAG {c['dato_rag'].mean():.2f} vs sin RAG {c['dato_sin_rag'].mean():.2f}; "
          f"similitud {c['sim_rag'].mean():.2f} vs {c['sim_sin_rag'].mean():.2f}; fidelidad (juez) {c['fidelidad_rag'].astype(float).mean():.1f} vs {c['fidelidad_sin_rag'].astype(float).mean():.1f}.")
    print(f"   - El contexto recuperado contenía el chunk correcto en {100*c['ctx_relevante'].astype(float).mean():.0f} % de los casos; las fuentes finales las añade el código.")
    print(f"   - Abstenciones indebidas (con respuesta pero dijo 'no encuentro'): {int(c['abstencion_indebida'].sum())} de {len(c)}.")
    print(f"\n3) SIN RESPUESTA (con umbral): rechazó {int(s['rechaza_correcto'].sum())} de {len(s)}.")
    if "df_neg" in globals():
        print(f"   - SIN umbral (el LLM por sí solo): rechazó {int(df_neg['rechaza_correcto'].sum())} de {len(df_neg)}; las cercanas son la prueba real (el umbral solo filtra lo claramente lejano).")
    print("\n4) LIMITACIONES: modelo generador de 3B, evaluación 'gold' por regex, juez LLM pequeño (usar rúbrica manual), "
          "~20 preguntas (1 pregunta ≈ 0.05), PDFs con ruido de extracción.")
    print("5) MEJORAS: re-ranking con un modelo cruzado, modelo generador mayor, tablas de concordancias entre norma y Constitución.")
borrador_rag()

BORRADOR DE CONCLUSIONES — RAG

1) RECUPERACIÓN (modo híbrido, chunk por artículo, 23 preguntas evaluables): Hit@1 0.74, Hit@3 0.87, Hit@5 0.91, MRR 0.81, Recall@5 0.81.
   - Preguntas cruzadas: cobertura@5 = 0.83 (fracción de documentos requeridos presentes en el contexto).
   - Ablación: mejor MRR 0.91 (fijo/800/hibrido); la configuración elegida (por artículo, híbrido) da 0.81. Con 23 preguntas, 1 pregunta ≈ 0.04 de MRR: la diferencia es ruido, y el chunk por artículo se elige porque da cita exacta (los chunks fijos pueden etiquetar mal el artículo).

2) GENERACIÓN (n = 23 con respuesta): dato clave RAG 0.89 vs sin RAG 0.32; similitud 0.80 vs 0.70; fidelidad (juez) 3.3 vs 2.0.
   - El contexto recuperado contenía el chunk correcto en 91 % de los casos; las fuentes finales las añade el código.
   - Abstenciones indebidas (con respuesta pero dijo 'no encuentro'): 2 de 23.

3) SIN RESPUESTA (con umbral): rechazó 5 de 7.
   - SIN umbral (el LLM por sí solo): rechazó 5 de 7; las cercanas

[48749:48749:1020/210433.568699:ERROR:chrome/browser/ui/views/user_education/impl/browser_user_education_interface_impl.cc:154] Attempting to show IPH IPH_DiscardRing before browser initialization complete; IPH will not be shown.
[48749:48749:1020/210433.603965:ERROR:chrome/browser/ui/views/user_education/impl/browser_user_education_interface_impl.cc:154] Attempting to show IPH IPH_DiscardRing before browser initialization complete; IPH will not be shown.
[48749:48749:1020/210433.607891:ERROR:chrome/browser/ui/views/user_education/impl/browser_user_education_interface_impl.cc:154] Attempting to show IPH IPH_DiscardRing before browser initialization complete; IPH will not be shown.
[48749:48749:1020/210433.622453:ERROR:chrome/browser/ui/views/user_education/impl/browser_user_education_interface_impl.cc:154] Attempting to show IPH IPH_DiscardRing before browser initialization complete; IPH will not be shown.
